# 📐 Statistical Inference — C1–C3

Notebook này dùng dữ liệu `commits_features.csv` (mỗi dòng là một khối code duy nhất) để kiểm định các câu hỏi nghiên cứu. Kết quả gồm test statistic, p-value, effect size và khoảng tin cậy.

**Diễn giải:** mức ý nghĩa α = 0,05. Các khối cùng repo không độc lập, nên kết luận dựa trên `p_value_cluster` và khoảng tin cậy tính với sai số chuẩn theo cụm repo. C1.3 và C1.4 dùng Mann–Whitney trên từng khối, chưa điều chỉnh theo cụm.

**Giới hạn của biến:** `has_recorded_change` là cận dưới của việc code bị sửa (ô trống nghĩa là không thu thập được). `days_to_modification` dựa trên `commit_date`, vốn chỉ là cận trên của ngày đưa code vào, nên C1.3 và C3.1 là kết quả phụ.

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf
warnings.filterwarnings('ignore')

DATA_PATH = Path('../data/processed/commits_features.csv')
df = pd.read_csv(DATA_PATH, low_memory=False)
for col in ['commit_date', 'change_commit_date']: df[col] = pd.to_datetime(df[col], utc=True, errors='coerce')
Y = 'has_recorded_change'
df[Y] = df[Y].astype(int)
df['days_to_modification'] = pd.to_numeric(df['days_to_modification'], errors='coerce')
ALPHA = 0.05
REPORT_DIR = Path('../reports'); REPORT_DIR.mkdir(exist_ok=True)
print(f'Đã tải {len(df):,} dòng × {df.shape[1]} cột, {df.repo.nunique():,} repo')

def cluster_ols(formula, data):
    # sai số chuẩn theo cụm repo vì các khối cùng repo không độc lập
    return smf.ols(formula, data=data).fit(cov_type='cluster', cov_kwds={'groups': pd.factorize(data['repo'])[0]})

Đã tải 21,498 dòng × 23 cột, 12,249 repo


## C1 — Hypothesis testing

In [2]:
def cramers_v(table, chi2):
    n = table.to_numpy().sum(); r, k = table.shape
    return np.sqrt((chi2 / n) / max(1, min(k - 1, r - 1)))

def chi_square_result(name, row, col=Y):
    table = pd.crosstab(df[row], df[col])
    chi2, p, dof, expected = stats.chi2_contingency(table)
    # Wald test đồng thời cho mọi mức của row, sai số chuẩn theo cụm repo
    p_cluster = cluster_ols(f'{col} ~ C({row})', df).f_pvalue
    return {'question': name, 'test': 'Chi-square', 'statistic': chi2, 'p_value': p, 'p_value_cluster': p_cluster, 'dof': dof, 'effect_size': cramers_v(table, chi2), 'significant_α_0.05': p_cluster < ALPHA}

def mann_whitney_result(name, x, group):
    a = df.loc[(df[group] == 0) & df[x].notna(), x]
    b = df.loc[(df[group] == 1) & df[x].notna(), x]
    u, p = stats.mannwhitneyu(a, b, alternative='two-sided')
    # rank-biserial correlation: 0 = no separation, ±1 = complete separation
    rbc = 2 * u / (len(a) * len(b)) - 1
    return {'question': name, 'test': 'Mann–Whitney U', 'statistic': u, 'p_value': p, 'dof': np.nan, 'effect_size': rbc, 'n_group_0': len(a), 'n_group_1': len(b), 'median_group_0': a.median(), 'median_group_1': b.median(), 'significant_α_0.05': p < ALPHA}

c1_results = []
c1_results += [chi_square_result('C1.1 language × has_recorded_change', 'language')]
c1_results += [chi_square_result('C1.2 year × has_recorded_change', 'year')]
# C1.3 so sánh theo language; chỉ dùng khoảng thời gian > 0 (kết quả phụ, xem phần đầu)
days_py = df.loc[(df.language == 'python') & df.days_to_modification.notna(), 'days_to_modification']
days_js = df.loc[(df.language == 'js') & df.days_to_modification.notna(), 'days_to_modification']
u, p = stats.mannwhitneyu(days_py, days_js, alternative='two-sided')
c1_results += [{'question': 'C1.3 days_to_modification: Python vs JS', 'test': 'Mann–Whitney U', 'statistic': u, 'p_value': p, 'effect_size': 2*u/(len(days_py)*len(days_js))-1, 'n_python': len(days_py), 'n_js': len(days_js), 'median_python': days_py.median(), 'median_js': days_js.median(), 'significant_α_0.05': p < ALPHA}]
c1_results += [mann_whitney_result('C1.4 code_lines: has_recorded_change False vs True', 'code_lines', Y)]
c1_results += [chi_square_result('C1.5 ai_tool × has_recorded_change', 'ai_tool')]
c1_table = pd.DataFrame(c1_results)
display(c1_table)
c1_table.to_csv(REPORT_DIR / 'statistical_c1_hypothesis_tests.csv', index=False)

,question,test,statistic,p_value,p_value_cluster,dof,effect_size,significant_α_0.05,n_python,n_js,median_python,median_js,n_group_0,n_group_1,median_group_0,median_group_1
0,C1.1 language × has_recorded_change,Chi-square,1.638702e+01,5.163775e-05,0.257002,1.0,0.027609,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,C1.2 year × has_recorded_change,Chi-square,6.442574e+01,3.399656e-13,0.000108,4.0,0.054743,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,C1.3 days_to_modification: Python vs JS,Mann–Whitney U,3.255397e+06,3.874126e-06,NaN,NaN,0.082502,True,4387.0,1371.0,3.918,1.935,NaN,NaN,NaN,NaN
3,C1.4 code_lines: has_recorded_change False vs ...,Mann–Whitney U,4.409888e+07,1.792869e-83,NaN,NaN,-0.159901,True,NaN,NaN,NaN,NaN,13998.0,7500.0,29.0,43.0
4,C1.5 ai_tool × has_recorded_change,Chi-square,1.668101e+02,2.135125e-33,0.012851,6.0,0.088087,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## C2 — A/B testing, effect size và 95% confidence interval

Outcome là tỷ lệ `has_recorded_change`. Hiệu hai tỷ lệ, khoảng tin cậy 95% và p-value lấy từ hồi quy `has_recorded_change ~ nhóm` với sai số chuẩn theo cụm repo. Cohen's h đo độ lớn chênh lệch.

Không có so sánh trước/sau ChatGPT: commit sớm nhất (3/12/2022) đã sau ngày ChatGPT ra mắt (30/11/2022).

In [3]:
def cohens_h(p1, p2):
    return 2*np.arcsin(np.sqrt(p1)) - 2*np.arcsin(np.sqrt(p2))

def ab_test(name, mask_a, mask_b):
    d = df[mask_a | mask_b].assign(group_a=mask_a.astype(int))
    res = cluster_ols(f'{Y} ~ group_a', d)
    pa, pb = d.loc[d.group_a.eq(1), Y].mean(), d.loc[d.group_a.eq(0), Y].mean()
    lo, hi = res.conf_int().loc['group_a']
    return {'comparison': name, 'group_a_n': int(d.group_a.sum()), 'group_b_n': int(d.group_a.eq(0).sum()), 'group_a_rate': pa, 'group_b_rate': pb, 'difference_a_minus_b': pa-pb, 'diff_95CI_cluster': f'[{lo:.4f}, {hi:.4f}]', 'cohens_h': cohens_h(pa,pb), 'p_value_cluster': res.pvalues['group_a'], 'significant_α_0.05': res.pvalues['group_a'] < ALPHA}

ab_results = [
    ab_test('C2.2 Python vs JS', df.language.eq('python'), df.language.eq('js')),
    ab_test('C2.3 Weekend vs weekday', df.is_weekend.eq(True), df.is_weekend.eq(False)),
]
ab_table = pd.DataFrame(ab_results)
display(ab_table)
ab_table.to_csv(REPORT_DIR / 'statistical_c2_ab_tests.csv', index=False)

,comparison,group_a_n,group_b_n,group_a_rate,group_b_rate,difference_a_minus_b,diff_95CI_cluster,cohens_h,p_value_cluster,significant_α_0.05
0,C2.2 Python vs JS,15781,5717,0.356821,0.32692,0.029902,"[-0.0218, 0.0816]",0.063053,0.256989,False
1,C2.3 Weekend vs weekday,4854,16644,0.335806,0.35268,-0.016874,"[-0.0708, 0.0370]",-0.035518,0.539576,False


## C3 — Bootstrap confidence intervals

C3.1 (median số ngày đến lần thay đổi đầu tiên) là kết quả phụ vì `commit_date` có thể muộn hơn ngày thật.

C3.2 dùng bootstrap theo cụm repo và có thêm:
- tỷ lệ khi bỏ `conradhuebler/ALIMA`, repo có nhiều khối và tỷ lệ thay đổi cao;
- tỷ lệ có thay đổi trong 30 ngày, chỉ tính các khối có ít nhất 30 ngày theo dõi, để các năm so sánh được với nhau. Khối có khoảng thời gian ≤ 0 bị loại vì không biết thời điểm thật.

In [4]:
rng = np.random.default_rng(42)
def bootstrap_stat(values, statistic=np.median, n_boot=5000):
    values = np.asarray(values, dtype=float); estimates = np.empty(n_boot)
    for i in range(n_boot): estimates[i] = statistic(rng.choice(values, size=len(values), replace=True))
    return float(statistic(values)), np.quantile(estimates, .025), np.quantile(estimates, .975)

def cluster_rate(d, col, n_boot=5000):
    # bootstrap theo cụm: lấy mẫu lại cả repo thay vì từng khối
    g = d.groupby('repo')[col].agg(['sum', 'size']).to_numpy(dtype=float)
    idx = rng.integers(0, len(g), size=(n_boot, len(g)))
    boot = g[idx, 0].sum(axis=1) / g[idx, 1].sum(axis=1)
    return g[:, 0].sum() / g[:, 1].sum(), *np.quantile(boot, [.025, .975])

valid_days = df.days_to_modification.dropna().to_numpy()
median, lo, hi = bootstrap_stat(valid_days, np.median)
bootstrap_median = pd.DataFrame([{'metric': 'median_days_to_modification', 'n': len(valid_days), 'estimate': median, 'ci_95_lower': lo, 'ci_95_upper': hi}])
print('C3.1 Bootstrap CI cho median days_to_modification (kết quả phụ):'); display(bootstrap_median)

OUTLIER_REPO = 'conradhuebler/ALIMA'
outlier = df.repo.eq(OUTLIER_REPO)
print(f'{OUTLIER_REPO}: {outlier.mean():.1%} số khối, {df.loc[outlier, Y].mean():.1%} có ghi nhận thay đổi')
# Mốc cuối quan sát là lần thay đổi muộn nhất trong dữ liệu
end = df.change_commit_date.max()
d30 = df[df.commit_date.le(end - pd.Timedelta(days=30)) & ~(df[Y].eq(1) & df.days_to_modification.isna())]
d30 = d30.assign(change_30d=d30.days_to_modification.le(30).astype(int))

year_rows = []
for year, group in df.groupby('year', dropna=True):
    rate, lo, hi = cluster_rate(group, Y)
    rate_x, lo_x, hi_x = cluster_rate(group[~group.repo.eq(OUTLIER_REPO)], Y)
    g30 = d30[d30.year.eq(year)]
    rate_30, lo_30, hi_30 = cluster_rate(g30, 'change_30d')
    year_rows.append({'year': int(year), 'n': len(group), 'n_repos': group.repo.nunique(), 'change_rate': rate, 'ci_95_lower': lo, 'ci_95_upper': hi,
                      'change_rate_no_alima': rate_x, 'no_alima_ci_95_lower': lo_x, 'no_alima_ci_95_upper': hi_x,
                      'n_30d': len(g30), 'change_rate_30d': rate_30, 'ci_30d_95_lower': lo_30, 'ci_30d_95_upper': hi_30})
bootstrap_year = pd.DataFrame(year_rows).sort_values('year')
print('C3.2 Tỷ lệ has_recorded_change theo năm (bootstrap theo cụm repo):'); display(bootstrap_year)
bootstrap_median.to_csv(REPORT_DIR / 'statistical_c3_median_bootstrap.csv', index=False)
bootstrap_year.to_csv(REPORT_DIR / 'statistical_c3_year_bootstrap.csv', index=False)

C3.1 Bootstrap CI cho median days_to_modification (kết quả phụ):


,metric,n,estimate,ci_95_lower,ci_95_upper
0,median_days_to_modification,5758,3.0865,2.902,3.653


conradhuebler/ALIMA: 7.8% số khối, 61.6% có ghi nhận thay đổi


C3.2 Tỷ lệ has_recorded_change theo năm (bootstrap theo cụm repo):


,year,n,n_repos,change_rate,ci_95_lower,ci_95_upper,change_rate_no_alima,no_alima_ci_95_lower,no_alima_ci_95_upper,n_30d,change_rate_30d,ci_30d_95_lower,ci_30d_95_upper
0,2022,55,44,0.490909,0.333333,0.659574,0.490909,0.333333,0.660377,53,0.396226,0.257565,0.553191
1,2023,1539,1117,0.438596,0.405691,0.470435,0.438596,0.405797,0.472130,1385,0.303249,0.273268,0.334068
2,2024,4001,2656,0.341165,0.313987,0.367233,0.341165,0.314576,0.366242,3654,0.237001,0.213839,0.259582
3,2025,11382,6227,0.340362,0.292281,0.400562,0.306005,0.286994,0.324365,10581,0.218410,0.199604,0.234963
4,2026,4521,2374,0.344835,0.288512,0.396790,0.318239,0.280842,0.361802,4081,0.222494,0.194594,0.248889


## Cách viết kết luận

Với mỗi câu hỏi, báo cáo test, `p_value_cluster`, effect size và kết luận. Không viết 'chứng minh nguyên nhân'; chỉ viết 'có/không có bằng chứng thống kê về sự khác biệt/liên hệ'. Bootstrap CI mô tả độ bất định của ước lượng, không thay thế cho kiểm định nhân quả.

Viết "có ghi nhận thay đổi" thay cho "bị sửa"/"không bị sửa", và coi tỷ lệ là cận dưới. Không kết luận về độ tin cậy của code AI vì không có nhóm đối chứng là code không dùng AI.